In [1]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import sys

# Add project root to path
project_root = Path('.').resolve().parent.parent
sys.path.insert(0, str(project_root))

from grelu.interpret import motifs
from grelu.sequence.format import one_hot_to_strings
from src.utils.config_loader import PathConfig
from src.application.cnn import SimpleCNN
from src.application.utils import *

/Users/marta/miniconda3/envs/baggls/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
paths = PathConfig()

pthresh = 1e-4
nr_test_sequences = 2000

# Load data from testdata directory
data_path = paths.get_path('testdata_dir')
test_data = np.load(data_path / 'testdata.npy')
test_labels = np.load(data_path / 'testlabels.npy')
test_pos = np.load(data_path / 'testpos.npy')

## FIMO

In [3]:
# Get JASPAR motif file path
motif_path = paths.get_path('external.jaspar_motif_file')
if motif_path is None or not motif_path.exists():
    raise FileNotFoundError(
        "JASPAR motif file not found. Please set 'external.jaspar_motif_file' "
        "in configs/paths.yml to point to your JASPAR motif file"
    )

In [4]:
testdata_strings = one_hot_to_strings(torch.tensor(test_data))

print(f"Running FIMO with p-threshold={pthresh}, this takes a couple of minutes...")
fimo_res = motifs.scan_sequences(seqs=testdata_strings, motifs=str(motif_path), pthresh=pthresh)

Running FIMO with p-threshold=0.0001, this takes a couple of minutes...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/Users/marta/miniconda3/envs/baggls/lib/python3.12/site-packages/tangermeme/tools/fimo.py:406: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hits = pandas.concat(hits)
/Users/marta/miniconda3/envs/baggls/lib/python3.12/site-packages/tangermeme/tools/fimo.py:406: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hits = pandas.concat(hits)
/Users/marta/miniconda3/envs/baggls/lib/python3.12/site-packages/ta

In [5]:
# Save FIMO results
output_dir = paths.get_path('xai_dir', mkdir=True)
output_dir.mkdir(parents=True, exist_ok=True)

fimo_output_path = output_dir / f'fimo_res_{pthresh}.csv'
fimo_res.to_csv(fimo_output_path)
fimo_res.shape

(141387, 9)

In [6]:
# Load FIMO results if already computed
output_dir = paths.get_path('xai_dir', mkdir=True)
fimo_output_path = output_dir / f'fimo_res_{pthresh}.csv'
fimo_res = pd.read_csv(fimo_output_path, index_col=0)
fimo_res.shape

(141387, 9)

## Attribution maps

In [7]:
# Load attribution map. It should have the shape(nr_seqs, 4, seq_length)
attribution_map = np.load(output_dir / 'attribution_map.npy')
attribution_map.shape

(2000, 4, 500)

In [8]:
attr_sum = np.sum(attribution_map, axis=1) # Not implemented for hypothetical contributions
attr_dm = avg_contributions(fimo_res, attr_sum)
attr_dm.to_csv(output_dir / f'attr_dm_{pthresh}.csv')

/Users/marta/miniconda3/envs/baggls/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/Users/marta/miniconda3/envs/baggls/lib/python3.12/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
